# Video Knowledge Extractor: GPU pipeline
Run top to bottom. Needs: GPU on, Internet on, the `GROQ_API_KEY` secret for the LLM step, the project attached as a dataset, and a demo video dataset. Audio speaker clustering uses SpeechBrain and downloads its public encoder on first use.

In [ ]:
!pip install -q faster-whisper==1.2.1 speechbrain sentence-transformers mediapipe==0.10.14 opencv-python-headless numpy==1.26.4 scikit-learn soundfile openai groq python-dotenv
!apt-get -qq install -y ffmpeg > /dev/null
# SpeechBrain downloads its public speaker encoder on first use.

In [ ]:
import os, sys, glob
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
for key in ['GROQ_API_KEY','GROQ_MODEL','LLM_API_KEY','LLM_BASE_URL','LLM_MODEL','AZURE_OPENAI_ENDPOINT','AZURE_OPENAI_API_KEY','AZURE_OPENAI_DEPLOYMENT']:
    try: os.environ[key] = s.get_secret(key)
    except Exception: pass
os.environ['JOBS_DIR'] = '/kaggle/working/jobs'

In [ ]:
# Point this at the attached project dataset (the folder that contains run_pipeline.py)
hits = glob.glob('/kaggle/input/**/run_pipeline.py', recursive=True)
print(hits)
sys.path.insert(0, os.path.dirname(hits[0]))

In [ ]:
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# Your demo videos (attached as a dataset)
videos = sorted(glob.glob('/kaggle/input/**/*.mp4', recursive=True))
print(*videos, sep='\n')

In [ ]:
from run_pipeline import process_video

# First run: audio speaker clustering + transcript + Groq analysis.
# Video face tracking and embeddings can be enabled in later runs.
process_video(videos[0], 'demo1', use_faces=False, use_llm=True, embed=False)

In [ ]:
# Check the output
import json
r = json.load(open('/kaggle/working/jobs/demo1/result.json'))
print(r['title'], '|', r['topic'][:200])
print([ (s['id'], s['name'], s['on_camera']) for s in r['speakers'] ])
for m in r['messages'][:8]: print(m['speaker'], round(m['start']), m['text'][:80])

In [ ]:
# Zip every finished job for download (Output tab) -> unzip into backend/jobs/ on Azure
!cd /kaggle/working && zip -rq jobs.zip jobs && ls -lh jobs.zip